# Q2: CycleGAN for Person Face Sketches
Unpaired translation between real faces (domain A) and sketches (domain B), following Zhu et al., *Unpaired Image-to-Image Translation using Cycle-Consistent Adversarial Networks* (ICCV 2017).

- G_AB: face -> sketch, G_BA: sketch -> face; D_A, D_B: 70x70 PatchGAN discriminators
- Loss = LSGAN adversarial + 10 x cycle-consistency (L1) + 5 x identity (L1)
- Adam (lr 2e-4, beta1 0.5), constant LR for the first half of training then linear decay to 0; image history pool of 50
- **Checkpoint saved after every epoch to Google Drive; the notebook resumes automatically from the last one.**

Run on Colab with a GPU runtime. Upload your `kaggle.json` when prompted.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import os
CKPT_DIR = '/content/drive/MyDrive/cyclegan_faces'   # checkpoints + final weights live here
os.makedirs(CKPT_DIR, exist_ok=True)

## 1. Dataset download
Download *Person Face Sketches* from Kaggle (needs `kaggle.json` from your Kaggle account > Settings > API).

In [ ]:
if not os.path.exists('/root/.kaggle/kaggle.json'):
    up = files.upload()                      # choose kaggle.json
    os.makedirs('/root/.kaggle', exist_ok=True)
    open('/root/.kaggle/kaggle.json', 'wb').write(up['kaggle.json']); os.chmod('/root/.kaggle/kaggle.json', 0o600)
if not os.path.isdir('/content/faces'):
    !kaggle datasets download -d almightyj/person-face-sketches -p /content/faces --unzip
!find /content/faces -maxdepth 3 -type d | head -20

## 2. Model definitions
(Identical to `app/models.py`, which the Flask app imports.)

In [ ]:
"""CycleGAN networks (ResNet generator + PatchGAN discriminator) shared by the notebook and the Flask app."""
import torch
import torch.nn as nn

IMG_SIZE = 256   # training / inference resolution
N_RES = 9        # residual blocks in the generator (9 for 256x256, as in the CycleGAN paper)


class ResnetBlock(nn.Module):
    """Residual block: conv-IN-ReLU-conv-IN with a skip connection and reflection padding."""
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1), nn.Conv2d(dim, dim, 3), nn.InstanceNorm2d(dim), nn.ReLU(True),
            nn.ReflectionPad2d(1), nn.Conv2d(dim, dim, 3), nn.InstanceNorm2d(dim))

    def forward(self, x):
        return x + self.block(x)


class Generator(nn.Module):
    """c7s1-64, d128, d256, R256 x n_res, u128, u64, c7s1-3 + Tanh (Johnson et al. style, as in CycleGAN)."""
    def __init__(self, in_ch=3, out_ch=3, ngf=64, n_res=N_RES):
        super().__init__()
        layers = [nn.ReflectionPad2d(3), nn.Conv2d(in_ch, ngf, 7), nn.InstanceNorm2d(ngf), nn.ReLU(True)]
        ch = ngf
        for _ in range(2):  # downsample
            layers += [nn.Conv2d(ch, ch * 2, 3, 2, 1), nn.InstanceNorm2d(ch * 2), nn.ReLU(True)]
            ch *= 2
        layers += [ResnetBlock(ch) for _ in range(n_res)]
        for _ in range(2):  # upsample
            layers += [nn.ConvTranspose2d(ch, ch // 2, 3, 2, 1, output_padding=1), nn.InstanceNorm2d(ch // 2), nn.ReLU(True)]
            ch //= 2
        layers += [nn.ReflectionPad2d(3), nn.Conv2d(ch, out_ch, 7), nn.Tanh()]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


class Discriminator(nn.Module):
    """70x70 PatchGAN: C64-C128-C256-C512 then a 1-channel patch score map."""
    def __init__(self, in_ch=3, ndf=64):
        super().__init__()
        def blk(i, o, s, norm=True):
            return [nn.Conv2d(i, o, 4, s, 1)] + ([nn.InstanceNorm2d(o)] if norm else []) + [nn.LeakyReLU(0.2, True)]
        self.model = nn.Sequential(*blk(in_ch, ndf, 2, False), *blk(ndf, ndf * 2, 2), *blk(ndf * 2, ndf * 4, 2),
                                   *blk(ndf * 4, ndf * 8, 1), nn.Conv2d(ndf * 8, 1, 4, 1, 1))

    def forward(self, x):
        return self.model(x)


def init_weights(net, std=0.02):
    """Initialise conv weights from N(0, 0.02) as in the paper."""
    for m in net.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.normal_(m.weight, 0.0, std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
    return net

## 3. Data pipeline
Images are classified into domains by folder name (`photo` -> A, `sketch` -> B), searched recursively so the exact Kaggle layout doesn't matter. Domains are sampled **independently** (unpaired). Preprocessing: resize 286, random crop 256, random flip, normalise to [-1, 1]. If memory is tight, set `MAX_PER_DOMAIN`.

In [ ]:
import glob, random, itertools, time
import numpy as np, torch, torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE, LR, EPOCHS = 1, 2e-4, 40
LAMBDA_CYC, LAMBDA_ID = 10.0, 5.0
MAX_PER_DOMAIN = None        # e.g. 800 to subsample if memory/time is a problem
EXTS = ('.jpg', '.jpeg', '.png')

def collect(root):
    """Return (photo_paths, sketch_paths) by inspecting the parent folder names."""
    A, B = [], []
    for p in glob.glob(os.path.join(root, '**', '*'), recursive=True):
        if p.lower().endswith(EXTS):
            d = os.path.dirname(p).lower()
            (B if 'sketch' in d else A if 'photo' in d else []).append(p)
    return sorted(A), sorted(B)

class UnpairedDataset(Dataset):
    """Pairs each domain-A image with a randomly chosen domain-B image (no correspondence)."""
    def __init__(self, A, B, train=True):
        self.A, self.B, self.train = A, B, train
        tf = [T.Resize(286, T.InterpolationMode.BICUBIC), T.RandomCrop(IMG_SIZE), T.RandomHorizontalFlip()] if train \
            else [T.Resize((IMG_SIZE, IMG_SIZE), T.InterpolationMode.BICUBIC)]
        self.tf = T.Compose(tf + [T.ToTensor(), T.Normalize((.5, .5, .5), (.5, .5, .5))])
    def __len__(self): return max(len(self.A), len(self.B))
    def __getitem__(self, i):
        a = Image.open(self.A[i % len(self.A)]).convert('RGB')
        b = Image.open(random.choice(self.B) if self.train else self.B[i % len(self.B)]).convert('RGB')
        return self.tf(a), self.tf(b)

A_paths, B_paths = collect('/content/faces')
print('faces:', len(A_paths), 'sketches:', len(B_paths))
assert A_paths and B_paths, 'Could not find photo/sketch folders - inspect the directory listing above.'
random.seed(0); random.shuffle(A_paths); random.shuffle(B_paths)
if MAX_PER_DOMAIN: A_paths, B_paths = A_paths[:MAX_PER_DOMAIN], B_paths[:MAX_PER_DOMAIN]
nA, nB = int(.9 * len(A_paths)), int(.9 * len(B_paths))
train_ds = UnpairedDataset(A_paths[:nA], B_paths[:nB], True)
test_ds = UnpairedDataset(A_paths[nA:], B_paths[nB:], False)
train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

## 4. Training utilities: image pool, LR schedule, checkpoint/resume

In [ ]:
class ImagePool:
    """History of 50 generated images; discriminators see a mix of new and old fakes (stabilises training)."""
    def __init__(self, size=50): self.size, self.imgs = size, []
    def query(self, batch):
        out = []
        for im in batch.detach():
            im = im.unsqueeze(0)
            if len(self.imgs) < self.size: self.imgs.append(im); out.append(im)
            elif random.random() < 0.5:
                j = random.randrange(self.size); out.append(self.imgs[j].clone()); self.imgs[j] = im
            else: out.append(im)
        return torch.cat(out)

G_AB, G_BA = init_weights(Generator()).to(DEVICE), init_weights(Generator()).to(DEVICE)   # face->sketch, sketch->face
D_A, D_B = init_weights(Discriminator()).to(DEVICE), init_weights(Discriminator()).to(DEVICE)
opt_G = torch.optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()), lr=LR, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(itertools.chain(D_A.parameters(), D_B.parameters()), lr=LR, betas=(0.5, 0.999))
decay_start = EPOCHS // 2
lr_lambda = lambda e: 1.0 if e < decay_start else max(0.0, 1.0 - (e - decay_start) / (EPOCHS - decay_start))
sched_G = torch.optim.lr_scheduler.LambdaLR(opt_G, lr_lambda); sched_D = torch.optim.lr_scheduler.LambdaLR(opt_D, lr_lambda)
pool_A, pool_B = ImagePool(), ImagePool()
mse, l1 = nn.MSELoss(), nn.L1Loss()
history = []; start_epoch = 0
CKPT = os.path.join(CKPT_DIR, 'last.pth')

def save_checkpoint(epoch):
    """Save everything needed to resume, plus standalone generator weights for the Flask app."""
    torch.save({'epoch': epoch, 'G_AB': G_AB.state_dict(), 'G_BA': G_BA.state_dict(), 'D_A': D_A.state_dict(), 'D_B': D_B.state_dict(),
                'opt_G': opt_G.state_dict(), 'opt_D': opt_D.state_dict(), 'sched_G': sched_G.state_dict(), 'sched_D': sched_D.state_dict(),
                'history': history}, CKPT + '.tmp')
    os.replace(CKPT + '.tmp', CKPT)                    # atomic: a crash mid-save never corrupts last.pth
    torch.save(G_AB.state_dict(), os.path.join(CKPT_DIR, 'G_face2sketch.pth'))
    torch.save(G_BA.state_dict(), os.path.join(CKPT_DIR, 'G_sketch2face.pth'))

if os.path.exists(CKPT):                              # resume
    ck = torch.load(CKPT, map_location=DEVICE)
    for n, m in [('G_AB', G_AB), ('G_BA', G_BA), ('D_A', D_A), ('D_B', D_B)]: m.load_state_dict(ck[n])
    for n, o in [('opt_G', opt_G), ('opt_D', opt_D), ('sched_G', sched_G), ('sched_D', sched_D)]: o.load_state_dict(ck[n])
    history, start_epoch = ck['history'], ck['epoch'] + 1
    print('Resumed from epoch', start_epoch)

## 5. Training loop
Per step: update generators (adversarial + cycle + identity), then discriminators (real vs pooled fakes, averaged loss x 0.5).

In [ ]:
def denorm(x): return (x * 0.5 + 0.5).clamp(0, 1)

@torch.no_grad()
def preview(epoch, n=3):
    """Show face -> sketch -> reconstructed face, and sketch -> face -> reconstructed sketch on held-out images."""
    G_AB.eval(); G_BA.eval()
    a, b = zip(*[test_ds[i] for i in range(n)]); a, b = torch.stack(a).to(DEVICE), torch.stack(b).to(DEVICE)
    rows = [a, G_AB(a), G_BA(G_AB(a)), b, G_BA(b), G_AB(G_BA(b))]
    titles = ['face', 'face->sketch', 'cycle', 'sketch', 'sketch->face', 'cycle']
    fig, ax = plt.subplots(6, n, figsize=(3 * n, 18))
    for r in range(6):
        for c in range(n):
            ax[r, c].imshow(denorm(rows[r][c]).permute(1, 2, 0).cpu()); ax[r, c].axis('off')
            if c == 0: ax[r, c].set_title(titles[r], loc='left')
    plt.suptitle(f'epoch {epoch}'); plt.tight_layout(); plt.show(); G_AB.train(); G_BA.train()

for epoch in range(start_epoch, EPOCHS):
    t0, agg = time.time(), np.zeros(4)
    for real_A, real_B in train_dl:
        real_A, real_B = real_A.to(DEVICE), real_B.to(DEVICE)
        # ---- generators ----
        fake_B, fake_A = G_AB(real_A), G_BA(real_B)
        pB, pA = D_B(fake_B), D_A(fake_A)
        loss_gan = mse(pB, torch.ones_like(pB)) + mse(pA, torch.ones_like(pA))
        loss_cyc = l1(G_BA(fake_B), real_A) + l1(G_AB(fake_A), real_B)
        loss_id = l1(G_AB(real_B), real_B) + l1(G_BA(real_A), real_A)
        loss_G = loss_gan + LAMBDA_CYC * loss_cyc + LAMBDA_ID * loss_id
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()
        # ---- discriminators ----
        fB, fA = pool_B.query(fake_B), pool_A.query(fake_A)
        rA, rB, dfA, dfB = D_A(real_A), D_B(real_B), D_A(fA), D_B(fB)
        loss_D = 0.5 * (mse(rA, torch.ones_like(rA)) + mse(dfA, torch.zeros_like(dfA))
                        + mse(rB, torch.ones_like(rB)) + mse(dfB, torch.zeros_like(dfB)))
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()
        agg += [loss_G.item(), loss_D.item(), loss_cyc.item(), loss_id.item()]
    sched_G.step(); sched_D.step()
    agg /= len(train_dl); history.append(agg.tolist())
    save_checkpoint(epoch)                                           # every epoch
    print(f'epoch {epoch+1}/{EPOCHS}  G {agg[0]:.3f}  D {agg[1]:.3f}  cycle {agg[2]:.3f}  id {agg[3]:.3f}  ({time.time()-t0:.0f}s)')
    if epoch % 5 == 0 or epoch == EPOCHS - 1: preview(epoch + 1)

In [ ]:
h = np.array(history)
for i, n in enumerate(['G total', 'D', 'cycle', 'identity']): plt.plot(h[:, i], label=n)
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.title('CycleGAN training losses'); plt.savefig('q2_losses.png', dpi=150); plt.show()

## 6. Testing: sketch -> face and face -> sketch on held-out images

In [ ]:
preview(EPOCHS, n=5)
print('Weights for the Flask app: G_face2sketch.pth and G_sketch2face.pth in', CKPT_DIR, '-> copy into app/weights/')

## Hyperparameter notes (fill in)
Record the learning rate / batch size / epochs tried and what you observed (e.g. batch size 1 with instance norm per the paper; lr 2e-4; 40 epochs; effect of lambda_cycle and identity loss).